# Violencia de pareja contra la mujer — ENDES 2021-2025

Cálculo reproducible de los indicadores de violencia familiar contra la mujer de 15 a 49 años
ejercida por el esposo o compañero, a partir de los microdatos de la **Encuesta Demográfica y
de Salud Familiar (ENDES)** del INEI.

Cada indicador se contrasta automáticamente contra el valor publicado en el
*Capítulo 12 · Violencia contra las mujeres, niñas y niños* del informe principal de la ENDES.
La sección [2. Validación](#2.-Validación-contra-los-cuadros-del-INEI) reporta esa comparación:
**55 valores de referencia (estimación puntual y desviación estándar, 2021-2025) coinciden con
la cifra oficial.**

| | |
|---|---|
| **Fuente** | INEI · ENDES, microdatos por año ([portal de microdatos](https://proyectos.inei.gob.pe/microdatos/)) |
| **Universo** | Mujeres de 15 a 49 años alguna vez unidas, seleccionadas y entrevistadas para el módulo de violencia |
| **Ponderación** | `V005 / 1 000 000` |
| **Inferencia** | Linealización de Taylor; estrato `V022`, conglomerado `V001` |

La lógica vive en el paquete [`src/endes_violencia`](../src/endes_violencia); este notebook solo
la orquesta y muestra resultados. La metodología detallada está en
[`docs/metodologia.md`](../docs/metodologia.md) y los resultados de la validación en
[`docs/validacion.md`](../docs/validacion.md).

## 0. Configuración

In [1]:
import sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

from endes_violencia import carga, config, descarga, indicadores, reportes, tabulacion, validacion

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

config.crear_directorios()
print("Años configurados:", ", ".join(map(str, config.ANIOS)))
print("Datos crudos     :", config.DIR_CRUDO)
print("Salidas          :", config.DIR_SALIDAS)

Años configurados: 2021, 2022, 2023, 2024, 2025
Datos crudos     : C:\Users\rodas\Desktop\Indicadores ENDES\data\raw
Salidas          : C:\Users\rodas\Desktop\Indicadores ENDES\outputs


## 1. Datos

### 1.1 Descarga

Se descargan únicamente los módulos que contienen las variables de violencia (~130 MB por año):

| Módulo | Archivo | Contenido |
|---|---|---|
| 1631 | `REC0111` | Características de la mujer, ponderador y variables de diseño |
| 1631 | `REC91` | Módulo Perú: etnicidad y búsqueda de ayuda institucional (`S1023*`) |
| 1635 | `RE516171` | Nupcialidad, actividad y empoderamiento (`V501`, `V502`, `V739`, `V743*`, `V744*`) |
| 1637 | `REC84DV` | Módulo de violencia doméstica (`D101*`, `D103*`, `D105*`, `D113`-`D121`, `QI1003*`) |

Para agregar un año nuevo basta con añadir su identificador de encuesta a
`config.ENCUESTAS`; el resto del flujo no cambia.

In [2]:
# La descarga es idempotente: no vuelve a bajar lo que ya está en data/raw.
descarga.descargar(config.ANIOS)

### 1.2 Consolidación

Se une por `CASEID` con validación `one_to_one` (una fila por mujer) y se apilan los años
en una sola base, guardada en parquet para no repetir la lectura de los `.dta`.

El año se guarda en una columna nueva, `anio`. Conviene **no** reutilizar `V000` para eso:
`V000` es el código de país-fase del recode DHS y sobrescribirlo rompe la trazabilidad
con la documentación de la encuesta.

In [3]:
if config.ARCHIVO_BASE.exists():
    base = carga.cargar_base()
else:
    base = carga.construir_base(config.ANIOS)

print(f"{len(base):,} filas × {base.shape[1]} columnas")
base.groupby("anio").size().rename("mujeres entrevistadas").to_frame()

188,469 filas × 240 columnas


,mujeres entrevistadas
anio,
2021,38635
2022,38105
2023,38352
2024,37117
2025,36260


### 1.3 Universo e indicadores

El universo de los cuadros 12.1 a 12.4 son las mujeres **alguna vez unidas** que fueron
**seleccionadas y entrevistadas** para el módulo de violencia:

* `V044` = *"Mujer seleccionada y entrevistada"*
* `V502` ≠ *"Nunca casada"*

#### Cómo se leen las categorías de respuesta

El módulo de violencia (`D103*`, `D105*`) no usa un simple Sí/No. Sus cuatro etiquetas son:

| Etiqueta | Significado | ¿Alguna vez? | ¿Últimos 12 meses? |
|---|---|:--:|:--:|
| `No` | Nunca ocurrió | — | — |
| `Nunca` | Ocurrió, pero **no** en los últimos 12 meses | ✔ | — |
| `Algunas veces` | Ocurrió algunas veces en los últimos 12 meses | ✔ | ✔ |
| `Frecuentemente` | Ocurrió frecuentemente en los últimos 12 meses | ✔ | ✔ |

La etiqueta `Nunca` cuenta como violencia en la ventana "alguna vez": es el
*"sí, pero no en el último año"*. El "no" verdadero es la categoría `No`.

In [4]:
df = indicadores.construir_todo(base)

resumen_universo = pd.DataFrame({
    "Mujeres entrevistadas": df.groupby("anio").size(),
    "En el universo (sin ponderar)": df.groupby("anio")["elegible"].sum(),
    "En el universo (ponderado)": (df.loc[df["elegible"]]
                                    .groupby("anio")["peso"].sum()),
})
resumen_universo

,Mujeres entrevistadas,En el universo (sin ponderar),En el universo (ponderado)
anio,,,
2021,38635,21797,"18,705.23"
2022,38105,21321,"18,397.60"
2023,38352,21349,"18,076.95"
2024,37117,20398,"17,435.53"
2025,36260,19271,"16,120.83"


## 2. Validación contra los cuadros del INEI

`validacion/valores_referencia_inei.csv` contiene los valores oficiales extraídos de los libros
de Excel del informe (cuadros 12.1, 12.1.1, 12.2, 12.4.1, 12.4.2 y 12.10). La tolerancia es de
0,005 puntos porcentuales cuando el INEI publica dos decimales y 0,05 cuando publica uno.

In [5]:
indicadores_validables = {
    **reportes.INDICADORES_ALGUNA_VEZ,
    **reportes.INDICADORES_12M,
    **reportes.AYUDA,
}
nacional = tabulacion.tabular(df, indicadores_validables)

comparacion = validacion.comparar(nacional)
validacion.informe(comparacion)

,cuadro,indicador,anio,valor_inei,valor_calculado,diferencia,ee_inei,ee_calculado,estado
0,12.1.1,Violencia física alguna vez,2021,26.70,26.69,-0.01,NaN,0.61,OK
1,12.1.1,Violencia física alguna vez,2022,27.76,27.76,-0.00,0.66,0.66,OK
2,12.1.1,Violencia física alguna vez,2023,27.19,27.19,-0.00,0.61,0.61,OK
3,12.1.1,Violencia física alguna vez,2024,25.46,25.46,0.00,0.64,0.64,OK
4,12.1.1,Violencia física alguna vez,2025,23.24,23.24,0.00,0.69,0.69,OK
5,12.1.1,Violencia psicológica y/o verbal alguna vez,2022,51.89,51.89,0.00,0.75,0.75,OK
6,12.1.1,Violencia psicológica y/o verbal alguna vez,2023,49.34,49.34,-0.00,0.72,0.72,OK
7,12.1.1,Violencia psicológica y/o verbal alguna vez,2024,48.42,48.42,0.00,0.75,0.75,OK
8,12.1.1,Violencia psicológica y/o verbal alguna vez,2025,45.60,45.60,-0.00,0.83,0.83,OK
9,12.1.1,Violencia sexual alguna vez,2021,5.90,5.86,-0.04,NaN,0.32,OK


In [6]:
print(validacion.resumen(comparacion).to_string(index=False))
print()
if validacion.todo_valida(comparacion):
    print(f"Los {len(comparacion)} valores de referencia coinciden con la cifra publicada.")
else:
    fuera = comparacion.loc[comparacion["estado"] != "OK"]
    print(f"{len(fuera)} valores fuera de tolerancia:")
    print(fuera[["cuadro", "variable", "anio", "valor_inei", "estimacion", "diferencia"]]
          .to_string(index=False))

       cuadro  OK
       12.1.1  18
12.1.1 / 12.1   1
12.1.1 / 12.2   1
        12.10  10
       12.4.1  15
       12.4.2  10

Los 55 valores de referencia coinciden con la cifra publicada.


### 2.1 Errores estándar

El INEI publica desviación estándar, intervalo de confianza al 95 % y coeficiente de variación
para cada estimación. Se reproducen con un estimador de linealización de Taylor sobre la razón
ponderada, tomando `V022` como estrato y `V001` como conglomerado.

In [7]:
ee = comparacion.loc[comparacion["ee_inei"].notna(),
                     ["indicador_inei", "anio", "ee_inei", "ee", "cv_inei"]].copy()
ee["cv_calculado"] = 100 * ee["ee"] / comparacion.loc[ee.index, "estimacion"]
ee["dif_ee"] = (ee["ee"] - ee["ee_inei"]).abs()
print(f"Diferencia máxima en la desviación estándar: {ee['dif_ee'].max():.4f} pp")
ee.rename(columns={"indicador_inei": "indicador", "ee_inei": "ee INEI",
                   "ee": "ee calculado", "cv_inei": "cv INEI"}).head(12)

Diferencia máxima en la desviación estándar: 0.0001 pp


,indicador,anio,ee INEI,ee calculado,cv INEI,cv_calculado,dif_ee
1,Violencia física alguna vez,2022,0.66,0.66,2.36,2.36,0.00
2,Violencia física alguna vez,2023,0.61,0.61,2.25,2.25,0.00
3,Violencia física alguna vez,2024,0.64,0.64,2.53,2.53,0.00
4,Violencia física alguna vez,2025,0.70,0.70,2.99,2.99,0.00
5,Violencia psicológica y/o verbal alguna vez,2022,0.75,0.75,1.45,1.45,0.00
6,Violencia psicológica y/o verbal alguna vez,2023,0.72,0.72,1.46,1.46,0.00
7,Violencia psicológica y/o verbal alguna vez,2024,0.75,0.75,1.55,1.55,0.00
8,Violencia psicológica y/o verbal alguna vez,2025,0.83,0.83,1.83,1.83,0.00
10,Violencia sexual alguna vez,2022,0.37,0.37,5.56,5.56,0.00
11,Violencia sexual alguna vez,2023,0.37,0.37,5.66,5.66,0.00


## 3. Prevalencia de la violencia de pareja

### 3.1 Serie nacional

In [8]:
tabla_alguna_vez = tabulacion.tabular(df, reportes.INDICADORES_ALGUNA_VEZ)
tabulacion.a_formato_ancho(tabla_alguna_vez)

indicador,anio,Violencia total alguna vez,Violencia psicológica y/o verbal alguna vez,Violencia física alguna vez,Violencia sexual alguna vez,Violencia física y/o sexual alguna vez
0,2021,54.86,50.83,26.69,5.86,27.28
1,2022,55.66,51.89,27.76,6.70,28.47
2,2023,53.76,49.34,27.19,6.46,27.88
3,2024,51.96,48.42,25.46,5.60,26.03
4,2025,49.09,45.60,23.24,5.64,23.67


In [9]:
tabla_12m = tabulacion.tabular(df, reportes.INDICADORES_12M)
tabulacion.a_formato_ancho(tabla_12m)

indicador,anio,"Violencia total, últimos 12 meses","Violencia psicológica y/o verbal, últimos 12 meses","Violencia física, últimos 12 meses","Violencia sexual, últimos 12 meses","Violencia física y/o sexual, últimos 12 meses"
0,2021,33.60,32.49,6.95,1.75,7.57
1,2022,35.63,34.81,8.12,2.17,8.57
2,2023,34.46,33.51,7.64,1.87,8.26
3,2024,33.89,33.13,7.00,1.71,7.51
4,2025,30.98,30.33,6.53,1.56,6.82


### 3.2 Precisión de las estimaciones

Un coeficiente de variación mayor a 15 % vuelve la estimación meramente referencial;
por encima de 50 % el INEI no la publica.

In [10]:
precision = tabla_alguna_vez[["anio", "indicador", "estimacion", "ee",
                              "ic_inf", "ic_sup", "cv", "n"]].copy()
precision.columns = ["Año", "Indicador", "Estimación", "D.E.", "IC 95% inf",
                     "IC 95% sup", "CV", "n"]
precision

,Año,Indicador,Estimación,D.E.,IC 95% inf,IC 95% sup,CV,n
0,2021,Violencia total alguna vez,54.86,0.68,53.53,56.19,1.24,21797
1,2021,Violencia psicológica y/o verbal alguna vez,50.83,0.68,49.50,52.16,1.34,21797
2,2021,Violencia física alguna vez,26.69,0.61,25.49,27.89,2.30,21797
3,2021,Violencia sexual alguna vez,5.86,0.32,5.24,6.48,5.38,21797
4,2021,Violencia física y/o sexual alguna vez,27.28,0.62,26.06,28.49,2.28,21797
5,2022,Violencia total alguna vez,55.66,0.75,54.19,57.13,1.35,21321
6,2022,Violencia psicológica y/o verbal alguna vez,51.89,0.75,50.41,53.36,1.45,21321
7,2022,Violencia física alguna vez,27.76,0.66,26.47,29.04,2.36,21321
8,2022,Violencia sexual alguna vez,6.70,0.37,5.97,7.43,5.56,21321
9,2022,Violencia física y/o sexual alguna vez,28.47,0.66,27.19,29.76,2.31,21321


### 3.3 Componentes de la violencia psicológica y/o verbal

La violencia psicológica y/o verbal es la unión de tres bloques (cuadro 12.2): situaciones de
control (`D101A`-`D101F`), situaciones humillantes (`D103A`) y amenazas (`D103B`, `D103D`).

El cuadro 12.2 del INEI publica cinco de los seis ítems de control, pero los seis entran en el
indicador agregado: excluir `D101D` deja la prevalencia ~0,3 pp por debajo de la cifra oficial.

In [11]:
componentes = tabulacion.tabular(df, reportes.COMPONENTES_PSICOLOGICA)
tabulacion.a_formato_ancho(componentes).T

,0,1,2,3,4
indicador,,,,,
anio,"2,021.00","2,022.00","2,023.00","2,024.00","2,025.00"
Es celoso o molesto si conversa con otro hombre,34.54,35.41,33.77,32.65,31.69
La acusa frecuentemente de ser infiel,13.71,14.42,14.45,14.24,12.93
Le impide que visite o la visiten sus amistades,11.40,13.13,11.59,11.45,11.33
Trata de limitar las visitas o el contacto con su familia,9.67,10.14,9.03,9.11,7.81
Insiste siempre en saber a dónde va,25.29,26.73,24.80,22.84,21.55
Desconfía con el dinero,10.87,11.03,10.82,11.14,9.59
Algún control,45.78,47.39,44.81,43.63,41.13
La ha humillado delante de los demás,17.12,17.51,18.10,17.28,15.59


### 3.4 Polivictimización

Categorías mutuamente excluyentes: cada mujer cuenta en una sola combinación.

In [12]:
poli = tabulacion.tabular(df, reportes.POLIVICTIMIZACION)
tabulacion.a_formato_ancho(poli)

indicador,anio,Física y psicológica (no sexual),Sexual y psicológica (no física),Física y sexual (no psicológica),"Física, sexual y psicológica"
0,2021,17.56,0.52,0.11,5.17
1,2022,18.23,0.63,0.13,5.85
2,2023,17.16,0.60,0.07,5.70
3,2024,17.04,0.54,0.11,4.92
4,2025,14.73,0.41,0.16,5.05


### 3.5 Desagregación por característica seleccionada

`tabulacion.tabular(..., por=...)` replica la estructura de los cuadros del INEI para
cualquier variable de corte. Las disponibles están en `reportes.CORTES`.

In [13]:
print(", ".join(f"{k} ({v})" for k, v in reportes.CORTES.items()))

Grupo de edad (V013), Área de residencia (V025), Región natural (V101), Departamento (V024), Nivel educativo (V149), Quintil de riqueza (V190), Estado conyugal (V501), Sexo del jefe del hogar (V151), Trabajó en los últimos 12 meses (V731), Lengua materna (S119), Autoidentificación étnica (S119D), El padre golpeaba a la madre (D121)


In [14]:
por_area = tabulacion.tabular(df, reportes.INDICADORES_ALGUNA_VEZ,
                              por="V025", con_error=False)
tabulacion.a_formato_ancho(por_area)

indicador,anio,V025,Violencia total alguna vez,Violencia psicológica y/o verbal alguna vez,Violencia física alguna vez,Violencia sexual alguna vez,Violencia física y/o sexual alguna vez
0,2021,Urbano,55.15,51.45,26.18,5.49,26.71
1,2021,Rural,53.78,48.55,28.56,7.24,29.37
2,2022,Urbano,56.21,52.49,28.15,6.76,28.83
3,2022,Rural,53.62,49.64,26.28,6.51,27.15
4,2023,Urbano,54.20,49.77,27.36,6.53,27.95
5,2023,Rural,52.18,47.78,26.57,6.22,27.63
6,2024,Urbano,52.60,49.20,25.49,5.50,26.04
7,2024,Rural,49.50,45.39,25.32,5.96,25.99
8,2025,Urbano,49.80,46.51,23.35,5.77,23.79
9,2025,Rural,46.49,42.28,22.84,5.16,23.25


In [15]:
por_edad = tabulacion.tabular(
    df, {"violencia_total_alguna_vez": "Total alguna vez",
         "violencia_total_12m": "Total últimos 12 meses"},
    por="V013", con_error=False)
tabulacion.a_formato_ancho(por_edad)

indicador,anio,V013,Total alguna vez,Total últimos 12 meses
0,2021,15-19,47.54,42.93
1,2021,20-24,48.62,35.57
2,2021,25-29,51.12,35.24
3,2021,30-34,54.57,34.22
4,2021,35-39,53.94,33.73
5,2021,40-44,60.75,34.64
6,2021,45-49,59.31,26.57
7,2022,15-19,51.02,45.35
8,2022,20-24,50.51,40.10
9,2022,25-29,50.16,36.10


## 4. Dinámicas y factores asociados

### 4.1 Consumo de bebidas alcohólicas de la pareja

In [16]:
alcohol = tabulacion.tabular(df, reportes.FACTORES)
display(tabulacion.a_formato_ancho(alcohol))

print("\nFrecuencia de consumo (D114) entre las mujeres cuya pareja consume alcohol:")
tabulacion.distribucion(df, "D114", universo="pareja_consume_alcohol")

indicador,anio,La pareja consume bebidas alcohólicas,La pareja consume algunas veces o con frecuencia
0,2021,72.99,56.43
1,2022,76.23,56.96
2,2023,80.31,59.53
3,2024,82.03,59.40
4,2025,81.77,59.59



Frecuencia de consumo (D114) entre las mujeres cuya pareja consume alcohol:


D114,anio,Algunas veces,Mucha frecuencia,Nunca
0,2021,68.91,8.40,22.69
1,2022,66.97,7.74,25.29
2,2023,66.52,7.61,25.87
3,2024,65.72,6.70,27.58
4,2025,65.96,6.92,27.12


### 4.2 Participación de la mujer en las decisiones del hogar

Indicador: la mujer tiene la última palabra **sola o junto con su esposo o compañero**
(cuadro 2.15).

> **Cuidado con las etiquetas.** ENDES 2021 rotula la decisión conjunta como `"Ambos"`,
> mientras que desde 2022 usa `"Entrevistada y esposo/compañero"`. Un filtro por subcadena
> `"Entrevistada"` pierde todo 2021 (p. ej. *grandes compras* caería de 84,5 % a 28,7 %) y a la
> vez suma indebidamente `"Entrevistada y otra persona"`. Por eso se comparan etiquetas completas
> contra la lista `indicadores.DECIDE_SOLA_O_CON_PAREJA`.

In [17]:
decisiones = tabulacion.tabular(df, reportes.TOMA_DECISIONES)
tabulacion.a_formato_ancho(decisiones)

indicador,anio,Gasto del propio dinero,Cuidado de su propia salud,Grandes compras del hogar,Compras para necesidades diarias,Visitas a familiares o amigos,Qué alimentos cocinar cada día
0,2021,97.37,90.95,84.53,92.52,90.52,93.80
1,2022,98.11,90.22,84.92,91.50,90.73,92.38
2,2023,97.92,91.19,85.51,91.40,91.66,91.42
3,2024,97.81,92.24,85.42,92.17,91.92,92.20
4,2025,97.65,92.71,85.71,92.38,92.47,91.55


### 4.3 Justificación de la violencia física

Universo: todas las mujeres entrevistadas a las que se formuló la pregunta.

In [18]:
justificacion = tabulacion.tabular(df, reportes.JUSTIFICA_VIOLENCIA)
tabulacion.a_formato_ancho(justificacion)

indicador,anio,Si sale sin decirle,Si descuida a los niños,Si ella discute con él,Si se niega a tener relaciones sexuales,Si ella quema la comida
0,2021,0.87,1.38,0.80,0.78,0.81
1,2022,0.57,1.07,0.56,0.49,0.53
2,2023,0.65,1.05,0.59,0.53,0.60
3,2024,0.53,0.89,0.49,0.48,0.51
4,2025,0.58,0.89,0.66,0.55,0.55


## 5. Búsqueda de ayuda

### 5.1 ¿Buscó ayuda? (cuadro 12.10)

Universo: mujeres a las que se preguntó si buscaron ayuda tras haber sido maltratadas
físicamente (`D119Y` no vacío). Ambos indicadores reproducen el cuadro 12.10 exactamente.

In [19]:
ayuda = tabulacion.tabular(df, reportes.AYUDA)
tabulacion.a_formato_ancho(ayuda)

indicador,anio,Buscó ayuda en personas cercanas,Buscó ayuda en alguna institución,Buscó ayuda en personas cercanas y/o en alguna institución
0,2021,44.02,29.27,53.98
1,2022,45.69,29.08,54.20
2,2023,45.34,29.66,54.76
3,2024,44.56,29.46,53.46
4,2025,45.47,31.36,56.02


### 5.2 ¿A quién pidió ayuda? (cuadro 12.11)

Universo: mujeres que buscaron ayuda en personas cercanas. Es una pregunta de respuesta
múltiple, así que los porcentajes no suman 100.

> `D119XI` ("otro pariente femenino") y `D119XJ` ("otro pariente masculino") son ambos
> parientes de la mujer. Clasificar `D119XJ` como *"otra persona"* infla esa categoría a
> ~7,5 % cuando el INEI publica ~1,7 %.

In [20]:
fuentes = tabulacion.tabular(df, reportes.FUENTES_AYUDA)
tabulacion.a_formato_ancho(fuentes).T

,0,1,2,3,4
indicador,,,,,
anio,"2,021.00","2,022.00","2,023.00","2,024.00","2,025.00"
Madre,41.22,37.08,39.01,37.74,40.56
Padre,13.91,16.62,14.93,16.38,16.51
Hermana,15.68,17.56,15.50,15.27,13.31
Hermano,11.35,11.66,10.64,13.42,10.41
Actual/último esposo o compañero,1.39,0.51,1.20,0.83,0.44
Suegros,9.20,8.85,10.04,7.09,9.18
Otro pariente del esposo,9.78,11.04,11.26,13.62,10.06
Otro pariente de la mujer,13.31,13.90,12.52,15.57,11.45


### 5.3 ¿A qué institución acudió? (cuadro 12.12)

Universo: mujeres que acudieron a alguna institución. `S1023AZ` se rotula
*"No, nunca ha buscado ayuda"*, por lo que la respuesta `"No"` identifica a quienes **sí**
acudieron.

In [21]:
instituciones = tabulacion.tabular(df, reportes.INSTITUCIONES_AYUDA)
tabulacion.a_formato_ancho(instituciones).T

,0,1,2,3,4
indicador,,,,,
anio,"2,021.00","2,022.00","2,023.00","2,024.00","2,025.00"
Comisaría,80.47,79.86,81.53,79.27,82.60
Juzgado,5.44,5.74,3.53,5.36,2.87
Fiscalía,6.64,6.93,6.52,5.46,5.43
Defensoría Municipal (DEMUNA),8.95,7.73,9.29,7.24,7.78
Ministerio de la Mujer y Poblaciones Vulnerables,6.91,8.45,8.96,7.13,8.84
Defensoría del Pueblo,0.80,0.83,1.22,1.20,1.56
Establecimiento de salud,3.99,5.27,5.77,8.32,5.52
Organización privada,0.30,0.09,0.67,0.01,0.06


### 5.4 Razones para no buscar ayuda (cuadro 12.13)

Distribución porcentual entre las mujeres que no buscaron ayuda.

In [22]:
tabulacion.distribucion(df, "D120").T

,0,1,2,3,4
D120,,,,,
anio,"2,021.00","2,022.00","2,023.00","2,024.00","2,025.00"
Ella tenía la culpa,2.82,2.46,1.73,1.48,1.58
Es parte de la vida,2.92,1.90,1.46,1.32,2.37
Miedo a que le pegara de neuvo a ella o a sus hijas e hijos,8.36,8.46,8.25,8.89,8.85
Miedo a que no le de dinero para el sustento de su familia,0.00,0.00,0.97,0.57,0.88
Miedo al divorcio/separación,3.19,2.94,2.82,3.06,3.56
Miedo de causarle un problema a la persona que le pego,5.64,6.40,5.85,4.82,4.54
No era necesario,42.87,44.13,44.96,43.61,42.97
No sabe donde ir/no conoce servicios,10.95,11.00,11.06,11.11,8.91


## 6. Exportación

Se generan tres productos en `outputs/`:

* `tablas/indicadores_violencia_endes.xlsx` — una hoja por tabla, más la hoja de validación.
* `csv/*.csv` — la misma información en formato largo, apta para BI o para volver a leer.
* `csv/microdatos_violencia.csv` — microdatos con los indicadores construidos y las variables
  de diseño, restringidos al universo, para modelar (regresiones, etc.).

In [23]:
hojas = reportes.construir_reportes(df)
informe_validacion = validacion.informe(comparacion)

salida_excel = config.DIR_TABLAS / "indicadores_violencia_endes.xlsx"
with pd.ExcelWriter(salida_excel, engine="openpyxl") as writer:
    informe_validacion.to_excel(writer, sheet_name="Validacion_INEI", index=False)
    for nombre, tabla in hojas.items():
        tabla.to_excel(writer, sheet_name=nombre[:31], index=False)

for nombre, tabla in hojas.items():
    tabla.to_csv(config.DIR_CSV / f"{nombre}.csv", index=False, encoding="utf-8-sig")

print(f"{len(hojas) + 1} hojas → {salida_excel.relative_to(RAIZ)}")
print("\n".join(f"  · {n}" for n in hojas))

25 hojas → outputs\tablas\indicadores_violencia_endes.xlsx
  · Nacional_alguna_vez
  · Nacional_12_meses
  · Polivictimizacion
  · Componentes_psicologica
  · Busqueda_de_ayuda
  · Fuentes_ayuda_cercanas
  · Instituciones_ayuda
  · Factores_alcohol
  · Toma_de_decisiones
  · Justifica_violencia
  · Por_Grupo_de_edad
  · Por_Area_de_residencia
  · Por_Region_natural
  · Por_Departamento
  · Por_Nivel_educativo
  · Por_Quintil_de_riqueza
  · Por_Estado_conyugal
  · Por_Sexo_del_jefe_del_hogar
  · Por_Trabajo_en_los_ultimos_12_
  · Por_Lengua_materna
  · Por_Autoidentificacion_etnica
  · Por_El_padre_golpeaba_a_la_mad
  · Dist_razones_no_ayuda
  · Dist_frecuencia_alcohol


In [24]:
columnas = ["anio", "CASEID", "peso", config.VAR_ESTRATO, config.VAR_CONGLOMERADO, "elegible"]
columnas += [c for c in df.columns
             if c.startswith(("violencia_", "poli_", "control_", "humillacion_", "amenazas_",
                              "ayuda_", "decide_", "justifica_", "pareja_consume_"))]
columnas += list(reportes.CORTES.values())
columnas = [c for c in dict.fromkeys(columnas) if c in df.columns]

ruta_micro = config.DIR_CSV / "microdatos_violencia.csv"
micro = df.loc[df["elegible"], columnas]
micro.to_csv(ruta_micro, index=False, encoding="utf-8-sig")
print(f"{len(micro):,} filas × {micro.shape[1]} columnas → {ruta_micro.relative_to(RAIZ)}")

104,136 filas × 58 columnas → outputs\csv\microdatos_violencia.csv


---

## Cómo citar

> Instituto Nacional de Estadística e Informática (INEI). *Encuesta Demográfica y de Salud
> Familiar (ENDES)*, 2021-2025. Microdatos. Lima, Perú.

El código de este repositorio se distribuye bajo licencia MIT; los microdatos y los cuadros
publicados son propiedad del INEI y se rigen por sus condiciones de uso.